In [1]:
!test -f obs_3day.bufr \
    || wget https://sites.ecmwf.int/repository/pdbufr/test-data/obs_3day.bufr \
             --output-document=obs_3day.bufr

# Flat reader: `required_columns` – block extraction mode

In [2]:
import warnings

import pdbufr

# disable warnings from the flat reader about non-overlapping columns
warnings.filterwarnings("ignore", module="pdbufr")

## Default behaviour (`required_columns=True` or `False`)

The default value is `True`, which in **block extraction mode** means that no columns are required and messages/subsets are always processed (supposing the filter conditions are met). The `required_columns=False` option has the same meaning in this mode.

In [3]:
# Default: no filtering – every message contributes at least one row
df_bool_true = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns="all",
    required_columns=True,
    reader="flat",
)

# False: no filtering – every message contributes at least one row
df_bool_false = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns="all",
    required_columns=False,
    reader="flat",
)

print(f"required_columns=True  → {len(df_bool_true)} rows, {len(df_bool_true.columns)} columns")
print(f"required_columns=False → {len(df_bool_false)} rows, {len(df_bool_false.columns)} columns")

required_columns=True  → 50 rows, 103 columns
required_columns=False → 50 rows, 103 columns


## Skipping messages that lack a key 

Setting `required_columns` to a key name (or a list of key names) causes messages that do not contain **all** of those keys to be skipped.

In [4]:
df_block = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns="all",
    required_columns="totalPrecipitationPast6Hours",
    reader="flat",
)
print(f"{len(df_block)} rows, {len(df_block.columns)} columns")
# Only the 6-hour precip column is present (24-hour precip messages were skipped)
assert "#1#totalPrecipitationPast6Hours" in df_block.columns
assert "#1#totalPrecipitationPast24Hours" not in df_block.columns

43 rows, 102 columns
